In [ ]:
# Default parameter values — overridden by papermill at runtime
run_id = "00000000-0000-0000-0000-000000000000"
input_s3_path = "s3://bucket/raw/uploads/1/sample.csv"
bucket = "django-prefect-datalake-dev"
aws_access_key_id = "rustfs"
aws_secret_access_key = "rustfs_secret"
aws_s3_region = "us-east-1"
s3_endpoint = "localhost:9000"
notebook_output_dir = "data/notebook_outputs"

In [ ]:
import json
import s3fs
import polars as pl

In [ ]:
s3 = s3fs.S3FileSystem(
    key=aws_access_key_id,
    secret=aws_secret_access_key,
    endpoint_url=f"http://{s3_endpoint}" if not s3_endpoint.startswith("http") else s3_endpoint,
    client_kwargs={"region_name": aws_s3_region},
)

In [ ]:
# Read raw file lazily — supports CSV and Parquet
if input_s3_path.endswith(".parquet"):
    df = pl.scan_parquet(input_s3_path, storage_options={
        "key": aws_access_key_id,
        "secret": aws_secret_access_key,
        "endpoint_url": f"http://{s3_endpoint}" if not s3_endpoint.startswith("http") else s3_endpoint,
    })
else:
    df = pl.scan_csv(input_s3_path, storage_options={
        "key": aws_access_key_id,
        "secret": aws_secret_access_key,
        "endpoint_url": f"http://{s3_endpoint}" if not s3_endpoint.startswith("http") else s3_endpoint,
    })

print(f"Schema: {df.schema}")

In [ ]:
output_s3_path = f"s3://{bucket}/processed/flows/data-processing/{run_id}/01_raw.parquet"

with s3.open(output_s3_path, "wb") as f:
    df.collect().write_parquet(f, compression="snappy", statistics=True, use_pyarrow=True)

row_count = df.select(pl.len()).collect().item()
print(f"Ingested {row_count} rows → {output_s3_path}")
print(json.dumps({"step": "ingest", "row_count": row_count, "output": output_s3_path}))